### Início
Gostaria de destacar inicialmente que, diante de um certo débito técnico da minha parte em relação à linguagem Python e suas especificidades, utilizei de IA para me auxiliar a entender de forma mais objetiva acerca do uso e da funcionalidades do Python nativo e a forma como Pydantic otimiza certas validações.

Os códigos das aulas que estou documentando estão presentes no repositório https://github.com/ArjanCodes/examples/tree/main/2024/pydantic_refresh disponibilizado no vídeo do card.

In [ ]:
from enum import auto, IntFlag
from typing import Any
from pydantic import BaseModel, EmailStr, Field, SecretStr, ValidationError

Nas importações, destaca-se que a importação do IntFlag e do Any traz ferramentas nativas da linguagem para lidar com operações lógicas e flexibilidade de tipos. Em seguida, as importações do Pydantic trazem os blocos de construção para o nosso sistema de validação: o molde principal (BaseModel), tipos semânticos (EmailStr, SecretStr), injetores de configuração (Field) e o capturador de falhas (ValidationError).

In [ ]:
class Role(IntFlag):
    Author = auto()
    Editor = auto()
    Developer = auto()
    Admin = Author | Editor | Developer

Aqui vemos a implementação da classe Role, que atribui os cargos e permissões do usuário. O `Int Flag` trata os valores como binários ao invés de textos isolados, com o `auto()` atribuindo esses valores matematicamente. O cargo 'Admin' atribui as permissões de determinada classe diante da necessidade do cargo; isso se deve ao uso do operador lógico "|" (OR, OU) que, diante do contexto, concede ao Admin a permissão de Author, Editor ou Developer.

In [ ]:
class User(BaseModel):
    name: str = Field(examples=["Arjan"])
    email: EmailStr = Field(
        examples=["example@arjancodes.com"],
        description="The email address of the user",
        frozen=True,
    )
    password: SecretStr = Field(
        examples=["Password123"], description="The password of the user"
    )
    role: Role = Field(default=None, description="The role of the user")

Na classe User, vemos o uso do `Field()`, função cujo uso é feito para enriquecer os atributos com metadados (exemples e description), fundamental para a geração automática de documentação de API's. Vemos o uso ds `SecrecStr`, que encripta a senha em forma de asteriscos na serialização, trazendo uma camada a mais de privacidade para a senha. Além disso, o uso de `frozen=True` no campo `email` impõem imutabilidade nos metadados ali presentes, impedindo qualquer alteração uma vez que o objeto User é instanciado na memória.

In [ ]:
def validate(data: dict[str, Any]) -> None:
    try:
        user = User.model_validate(data)
        print(user)
    except ValidationError as e:
        print("User is invalid")
        for error in e.errors():
            print(error)

Eis aqui uma das magias do Pydantic. A função `validate` recebe a variável `data`, cujo consiste no JSON contendo os devidos campos como Email, Name, password, description, etc. A partir disso, essa função varre o JSON buscando essas chaves e avalia se existe alguma irregularidade nos dados dentro das chaves; tal comparação é feita com base no `model_validate`. Caso exista, a função retorna um `Validation Error`.

In [ ]:
def main() -> None:
    good_data = {
        "name": "Arjan",
        "email": "example@arjancodes.com",
        "password": "Password123",
    }
    bad_data = {"email": "<bad data>", "password": "<bad data>"}

    validate(good_data)
    validate(bad_data)

if __name__ == "__main__":
    main()

Por fim, o código orquestra um teste prático. Ele cria dois pacotes de dados simulados (um íntegro e um corrompido) e os injeta no motor de validação.
O bloco if `__name__ == "__main__":` é um padrão fundamental no Python. Ele diz ao interpretador: "Execute a função main() apenas se este arquivo estiver sendo o ator principal (rodado diretamente no terminal)". Se este arquivo for importado por outro módulo do seu projeto apenas para usar a classe User, esses testes não serão executados indevidamente.